In [ ]:
import deap

In [ ]:
cxpb = 0.85

In [ ]:
from deap import base, creator

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

In [ ]:
def evalTSP(individual):
    distance = sum(DIST_MATRIX[individual[i]][individual[i+1]] for i in range(len(individual)-1))
    distance += DIST_MATRIX[individual[-1]][individual[0]]
    return distance,

In [ ]:
def cxOrdered(ind1, ind2):
    """Executes an ordered crossover (OX1) on the input individuals.
    The two individuals are modified in place.
    """
    size = len(ind1)
    a, b = sorted(random.sample(range(size), 2))
    
    holes1 = [True] * size
    holes2 = [True] * size
    
    for i in range(size):
        if i < a or i > b:
            holes1[ind2[i]] = False
            holes2[ind1[i]] = False
            
    temp1 = ind1[a:b+1]
    temp2 = ind2[a:b+1]
    
    k1, k2 = b + 1, b + 1
    for i in range(size):
        if not holes1[ind2[(b + 1 + i) % size]]:
            ind1[k1 % size] = ind2[(b + 1 + i) % size]
            k1 += 1
        if not holes2[ind1[(b + 1 + i) % size]]:
            ind2[k2 % size] = ind1[(b + 1 + i) % size]
            k2 += 1

    for i in range(a, b + 1):
        ind1[i], ind2[i] = temp2[i - a], temp1[i - a]

    return ind1, ind2

In [ ]:
toolbox.register("mutate", tools.mutShuffleIndexes, indpb=0.05)

In [ ]:
toolbox.register("select", tools.selTournament, tournsize=3)

In [ ]:
import random
from deap import base, creator, tools

IND_SIZE = 50  # Number of cities, adjust as needed

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
toolbox.register("indices", random.sample, range(IND_SIZE), IND_SIZE)
toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.indices)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
import random
import numpy as np
from deap import base, creator, tools

toolbox = base.Toolbox()

toolbox.register("indices", random.sample, range(creator.IND_SIZE), creator.IND_SIZE)
toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.indices)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def eval_tsp(individual, distance_matrix):
    distance = distance_matrix[individual[-1]][individual[0]]
    for gene1, gene2 in zip(individual[:-1], individual[1:]):
        distance += distance_matrix[gene1][gene2]
    return distance,

toolbox.register("evaluate", eval_tsp)
toolbox.register("mate", tools.cxOrdered)
toolbox.register("mutate", tools.mutShuffleIndexes, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

In [ ]:
import algorithms

def main_algorithm(pop, toolbox, cxpb, mutpb, ngen, stats=None, halloffame=None, verbose=True):
    return algorithms.eaSimple(pop, toolbox, cxpb, mutpb, ngen, stats=stats, halloffame=halloffame, verbose=verbose)

In [ ]:
import numpy as np
from deap import tools

stats = tools.Statistics(key=lambda ind: ind.fitness.values[0])
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)

hof = tools.HallOfFame(1)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_results(best_individual, cities_coords, logbook):
    """
    Plots the best TSP route and the convergence curve.
    
    Parameters:
    - best_individual: list of city indices representing the best route.
    - cities_coords: numpy array of shape (n_cities, 2) with x, y coordinates.
    - logbook: DEAP logbook object containing 'gen', 'min', 'avg', 'max' fitness stats.
    """
    # Close the route for plotting (return to start)
    route_indices = list(best_individual) + [best_individual[0]]
    route_coords = cities_coords[route_indices]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # --- Plot 1: Best Route ---
    ax1.plot(route_coords[:, 0], route_coords[:, 1], 'b-', linewidth=1.5, label='Route', zorder=1)
    ax1.scatter(cities_coords[:, 0], cities_coords[:, 1], c='red', s=50, zorder=2, label='Cities')
    
    # Annotate start city
    start_coord = cities_coords[best_individual[0]]
    ax1.scatter(start_coord[0], start_coord[1], c='green', s=150, marker='*', zorder=3, label='Start/End')
    
    ax1.set_title(f"Best Route (Distance: {np.sum(np.linalg.norm(np.diff(route_coords, axis=0), axis=1)):.2f})")
    ax1.set_xlabel("X Coordinate")
    ax1.set_ylabel("Y Coordinate")
    ax1.legend()
    ax1.grid(True, linestyle='--', alpha=0.7)
    ax1.set_aspect('equal', adjustable='box')
    
    # --- Plot 2: Convergence Curve ---
    gen = logbook.select("gen")
    fit_mins = logbook.select("min")
    fit_avgs = logbook.select("avg")
    fit_maxs = logbook.select("max")
    
    ax2.plot(gen, fit_mins, "b-", label="Minimum Fitness (Best Distance)")
    ax2.plot(gen, fit_avgs, "g--", label="Average Fitness")
    ax2.plot(gen, fit_maxs, "r:", label="Maximum Fitness (Worst Distance)")
    
    ax2.set_title("Convergence Curve")
    ax2.set_xlabel("Generation")
    ax2.set_ylabel("Fitness (Total Distance)")
    ax2.legend()
    ax2.grid(True, linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()

# Example Usage (assuming variables exist from previous DEAP execution):
# plot_results(best_ind, city_coords, log)